# Лабораторна робота №2 — Частина 1

У цій частині виконується завантаження та аналіз даних NOAA VHI/VCI для адміністративних одиниць України.

Основні дії:
1. Завантаження даних для кожної області.
2. Збереження файлів із датою та часом завантаження.
3. Захист від повторного завантаження.
4. Очищення даних та створення `pandas dataframe`.
5. Заміна індексів NOAA на індекси областей за українською абеткою.
6. Формування вибірок окремими функціями.

**Студент:** Сапронов Анатолій  
**Група:** ФБ-45


In [ ]:
import pandas as pd
import numpy as np
import requests
from pathlib import Path
from datetime import datetime
from io import StringIO

DATA_DIR = Path("data_noaa")
DATA_DIR.mkdir(exist_ok=True)

BASE_URL = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php"

# NOAA використовує власну нумерацію ProvinceID.
# Для лабораторної достатньо завантажити всі доступні id з цього діапазону.
PROVINCE_IDS = list(range(1, 28))

START_YEAR = 1981
END_YEAR = 2024


## 1. Завантаження даних NOAA

Файли зберігаються окремо для кожної області.  
У назві файлу є `provinceID` та дата/час завантаження.

Механізм захисту від повторного завантаження: якщо файл для області вже є у папці, він не завантажується повторно.


In [ ]:
def download_vhi_for_province(province_id: int, start_year: int = START_YEAR, end_year: int = END_YEAR) -> Path:
    existing_files = sorted(DATA_DIR.glob(f"vhi_province_{province_id}_*.csv"))
    if existing_files:
        print(f"Province {province_id}: файл вже існує, повторне завантаження не потрібне.")
        return existing_files[-1]

    params = {
        "country": "UKR",
        "provinceID": province_id,
        "year1": start_year,
        "year2": end_year,
        "type": "Mean"
    }

    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = DATA_DIR / f"vhi_province_{province_id}_{timestamp}.csv"
    file_path.write_text(response.text, encoding="utf-8")

    print(f"Province {province_id}: завантажено -> {file_path}")
    return file_path


downloaded_files = [download_vhi_for_province(pid) for pid in PROVINCE_IDS]
len(downloaded_files)


## 2. Зчитування та очищення даних

Під час очищення:
- прибираються зайві HTML-теги;
- назви стовпців приводяться до нормального вигляду;
- значення `VHI` переводяться у числовий формат;
- некоректні рядки видаляються.


In [ ]:
def read_vhi_file(file_path: Path, province_id: int) -> pd.DataFrame:
    text = file_path.read_text(encoding="utf-8", errors="ignore")
    text = text.replace("<pre>", "").replace("</pre>", "")
    text = text.replace("<br>", "\n").replace("</br>", "\n")

    df = pd.read_csv(StringIO(text), sep=r"[,;\s]+", engine="python")
    df.columns = [str(col).strip().replace("<", "").replace(">", "") for col in df.columns]

    # Стандартизація назв стовпців
    rename_map = {}
    for col in df.columns:
        c = col.lower()
        if "year" in c:
            rename_map[col] = "Year"
        elif "week" in c:
            rename_map[col] = "Week"
        elif "smi" in c:
            rename_map[col] = "SMN"
        elif "smt" in c:
            rename_map[col] = "SMT"
        elif "vci" in c and "tci" not in c:
            rename_map[col] = "VCI"
        elif "tci" in c:
            rename_map[col] = "TCI"
        elif "vhi" in c:
            rename_map[col] = "VHI"

    df = df.rename(columns=rename_map)

    needed = ["Year", "Week", "SMN", "SMT", "VCI", "TCI", "VHI"]
    df = df[[col for col in needed if col in df.columns]].copy()

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["Year", "Week", "VHI"])
    df["Year"] = df["Year"].astype(int)
    df["Week"] = df["Week"].astype(int)
    df["province_id_noaa"] = province_id

    return df


frames = []
for path in downloaded_files:
    pid = int(path.name.split("_")[2])
    try:
        frames.append(read_vhi_file(path, pid))
    except Exception as e:
        print(f"Не вдалося прочитати {path}: {e}")

vhi_df = pd.concat(frames, ignore_index=True)
vhi_df.head()


## 3. Заміна індексів областей

Нижче створено відповідність між назвами областей за українською абеткою та новими індексами.

Оскільки NOAA має власний порядок `provinceID`, у таблиці зберігаються обидва значення:
- `province_id_noaa` — індекс з NOAA;
- `province_index_ua` — індекс області за українською абеткою.


In [ ]:
ukrainian_regions_alphabetical = [
    "Автономна Республіка Крим",
    "Вінницька область",
    "Волинська область",
    "Дніпропетровська область",
    "Донецька область",
    "Житомирська область",
    "Закарпатська область",
    "Запорізька область",
    "Івано-Франківська область",
    "Київська область",
    "Кіровоградська область",
    "Луганська область",
    "Львівська область",
    "Миколаївська область",
    "Одеська область",
    "Полтавська область",
    "Рівненська область",
    "Сумська область",
    "Тернопільська область",
    "Харківська область",
    "Херсонська область",
    "Хмельницька область",
    "Черкаська область",
    "Чернівецька область",
    "Чернігівська область",
    "м. Київ",
    "м. Севастополь"
]

ua_index_map = {
    noaa_id: idx + 1
    for idx, noaa_id in enumerate(PROVINCE_IDS)
}

ua_name_map = {
    idx + 1: name
    for idx, name in enumerate(ukrainian_regions_alphabetical)
}

vhi_df["province_index_ua"] = vhi_df["province_id_noaa"].map(ua_index_map)
vhi_df["region_name_ua"] = vhi_df["province_index_ua"].map(ua_name_map)

vhi_df.head()


## 4. Функції для формування вибірок

Усі процедури реалізовані окремими функціями:
- ряд VHI для області за рік;
- ряд VHI за діапазон років;
- пошук екстремумів;
- роки екстремальної та помірної посухи.


In [ ]:
def get_vhi_series_for_year(df: pd.DataFrame, province_index_ua: int, year: int) -> pd.DataFrame:
    result = df[
        (df["province_index_ua"] == province_index_ua) &
        (df["Year"] == year)
    ][["Week", "VHI"]].sort_values("Week")
    return result


def get_vhi_series_for_year_range(df: pd.DataFrame, province_index_ua: int, start_year: int, end_year: int) -> pd.DataFrame:
    result = df[
        (df["province_index_ua"] == province_index_ua) &
        (df["Year"].between(start_year, end_year))
    ][["Year", "Week", "VHI"]].sort_values(["Year", "Week"])
    return result


def get_vhi_extremes(df: pd.DataFrame, province_index_ua: int, years: list[int]) -> pd.DataFrame:
    subset = df[
        (df["province_index_ua"] == province_index_ua) &
        (df["Year"].isin(years))
    ]

    result = subset.groupby("Year")["VHI"].agg(
        min_vhi="min",
        max_vhi="max",
        mean_vhi="mean",
        median_vhi="median"
    ).reset_index()

    return result


def get_drought_years(df: pd.DataFrame, province_index_ua: int, drought_type: str = "extreme") -> pd.DataFrame:
    subset = df[df["province_index_ua"] == province_index_ua].copy()

    if drought_type == "extreme":
        condition = subset["VHI"] < 15
    elif drought_type == "moderate":
        condition = (subset["VHI"] >= 15) & (subset["VHI"] < 35)
    else:
        raise ValueError("drought_type має бути 'extreme' або 'moderate'")

    result = (
        subset[condition]
        .groupby("Year")
        .agg(
            weeks_count=("Week", "count"),
            min_vhi=("VHI", "min"),
            mean_vhi=("VHI", "mean")
        )
        .reset_index()
        .sort_values("Year")
    )

    return result


## 5. Приклади виклику функцій

In [ ]:
# Приклад: область з індексом 1 за 2020 рік
get_vhi_series_for_year(vhi_df, province_index_ua=1, year=2020).head()


In [ ]:
# Приклад: VHI для області за 2020-2022 роки
get_vhi_series_for_year_range(vhi_df, province_index_ua=1, start_year=2020, end_year=2022).head()


In [ ]:
# Приклад: екстремуми для вказаних років
get_vhi_extremes(vhi_df, province_index_ua=1, years=[2019, 2020, 2021, 2022])


In [ ]:
# Приклад: роки екстремальної посухи
get_drought_years(vhi_df, province_index_ua=1, drought_type="extreme").head()


In [ ]:
# Приклад: роки помірної посухи
get_drought_years(vhi_df, province_index_ua=1, drought_type="moderate").head()
